# Retrieval Backend Evaluation: Vector vs. Hybrid vs. GraphRAG

This notebook compiles the MLflow automated benchmark results produced by the Hermes Agent and completes the final **GraphRAG** evaluation using the remote Memgraph deployment.

### 1. Hermes Agent Benchmark Results (Dense vs. Hybrid)
The Hermes Agent autonomously discovered that pure BM25/Lexical vectors dropped performance. When combined with a Cross-Encoder Reranker, Hybrid Search achieved a massive **60% reduction in retrieval misses**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Results exported from Hermes Agent Automation (legal-rag-bench)
data = {
    "Backend Config": ["Hybrid Baseline", "Dense", "Dense + Reranker", "Hybrid + Reranker (Winner)"],
    "nDCG@5": [0.289, 0.527, 0.567, 0.594],
    "Recall@5": [0.500, 0.667, 0.800, 0.800],
    "Miss Rate": [0.500, 0.333, 0.200, 0.200]
}

df_results = pd.DataFrame(data)
display(df_results)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_results, x="Backend Config", y="nDCG@5", palette="viridis")
plt.title("nDCG@5 Retrieval Performance (Higher is Better)")
plt.xticks(rotation=15)
plt.show()

### 2. GraphRAG Execution Environment
Hermes paused graph evaluation because no local Graph DB was running. We will now inject the remote Memgraph credentials provided by the infrastructure team to complete the triad.

In [ ]:
import os

# ⚠️ INSTRUCTIONS: Replace with the actual WhatsApp credentials!
os.environ["MEMGRAPH_USER"] = "hpi-project"
os.environ["MEMGRAPH_PASSWORD"] = "the password"  # <--- REPLACE
os.environ["MEMGRAPH_URL"] = "bolt://<the ip address>:7687" # <--- REPLACE

### 3. Generate Graph Config & Run Evaluation
We use the `deepfix` CLI to dynamically swap the backend to Property Graph and trigger the index/evaluation.

In [ ]:
!python -m deepfix_rag.main mutate-config --base configs/optimized.yaml --set retrieval.backend=graph --set retrieval.graph_store=memgraph --output configs/graph_candidate.yaml
!python -m deepfix_rag.main index --config configs/graph_candidate.yaml --paradigm graph
!python -m deepfix_rag.main eval --config configs/graph_candidate.yaml --compare configs/optimized.yaml